KWS Model Convertion from Tensorflow to C Header

[1] Block to inspect a specific model and verify if the structure is correct

In [ ]:
import tensorflow as tf
import numpy as np
from tabulate import tabulate

# Function to print details of the model

def inspect_tflite_model(model_path):
    # 1. Accessing to the model passed in function
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    # 2. Get information of the model
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    tensor_details = interpreter.get_tensor_details()
    ops = interpreter._get_ops_details()

    print("\n" + "="*50)
    print(f"Model Inspection: {model_path}")
    print("="*50 + "\n")

    # 3. Print Input Details
    print("\n[Input Tensors]")
    input_table = []
    for i, detail in enumerate(input_details):
        input_table.append([
            i, detail['name'], detail['shape'], detail['dtype'],
            detail.get('quantization', 'N/A')
        ])
    print(tabulate(input_table, headers=["Index", "Name", "Shape", "DType", "Quantization"]))

    # 4. Print Output Details
    print("\n[Output Tensors]")
    output_table = []
    for i, detail in enumerate(output_details):
        output_table.append([
            i, detail['name'], detail['shape'], detail['dtype'],
            detail.get('quantization', 'N/A')
        ])
    print(tabulate(output_table, headers=["Index", "Name", "Shape", "DType", "Quantization"]))

    # 5. Print Intermediate layers
    print("\n[All Tensors]")
    tensor_table = []
    for tensor in tensor_details:
        tensor_table.append([
            tensor['index'], tensor['name'], tensor['shape'], tensor['dtype'],
            tensor.get('quantization', 'N/A'), tensor.get('sparsity_parameters', 'N/A')
        ])
    print(tabulate(tensor_table, headers=["Index", "Name", "Shape", "DType", "Quantization", "Sparsity"]))

    # 6. Print Operations (How the layers are connected)
    print("\n[Operations]")
    ops_table = []
    for op in ops:
        ops_table.append([
            op['index'], op['op_name'], op['inputs'], op['outputs']
        ])
    print(tabulate(ops_table, headers=["Index", "Op Name", "Inputs", "Outputs"]))

    print("\n[Model Summary]")
    print(f"- Inputs: {len(input_details)}")
    print(f"- Outputs: {len(output_details)}")
    print(f"- Tensors: {len(tensor_details)}")
    print(f"- Operations: {len(ops)}")

if __name__ == "__main__":
    model_path = "kws_sheila_model.tflite"
    inspect_tflite_model(model_path)


Model Inspection: kws_sheila_model.tflite


[Input Tensors]
  Index  Name                 Shape        DType                    Quantization
-------  -------------------  -----------  -----------------------  --------------
      0  serving_default_x:0  [   1 1600]  <class 'numpy.float32'>  (0.0, 0)

[Output Tensors]
  Index  Name                       Shape    DType                    Quantization
-------  -------------------------  -------  -----------------------  --------------
      0  StatefulPartitionedCall:0  [1 2]    <class 'numpy.float32'>  (0.0, 0)

[All Tensors]
  Index  Name                                                                          Shape        DType                    Quantization    Sparsity
-------  ----------------------------------------------------------------------------  -----------  -----------------------  --------------  ----------
      0  serving_default_x:0                                                           [   1 1600]  <class 'numpy.fl

[2] Block to save data in a director

In [ ]:
import numpy as np
import tensorflow as tf
import os

# 1. Allocate Model
model_path = 'kws_sheila_model.tflite'
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# 2. Directory creation
output_dir = "tflite_tensors"
os.makedirs(output_dir, exist_ok=True)

#3. Inspection of each layer and saving file inside the folder

tensor_details = interpreter.get_tensor_details()

for tensor in tensor_details:
    try:
        if interpreter.get_tensor(tensor['index']).size == 0:
            print(f"Skipping empty tensor: {tensor['name']} (index {tensor['index']})")
            continue

        tensor_data = interpreter.get_tensor(tensor['index'])
        tensor_name = tensor['name'].replace('/', '_')

        filename = f"tensor_{tensor['index']:03d}_{tensor_name}_shape-{tensor['shape']}_dtype-{tensor['dtype']}.npy"
        np.save(os.path.join(output_dir, filename), tensor_data)
        print(f"Saved: {filename}")

    except ValueError as e:
        print(f"Failed to extract {tensor['name']} (index {tensor['index']}): {str(e)}")
        continue

print(f"\nExtraction complete. Output directory: {output_dir}/")


Saved: tensor_000_serving_default_x:0_shape-[   1 1600]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_001_sequential_dense_1_BiasAdd_ReadVariableOp_shape-[256]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_002_sequential_dense_2_BiasAdd_ReadVariableOp_shape-[256]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_003_sequential_dense_BiasAdd_ReadVariableOp_shape-[256]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_004_sequential_y_pred_BiasAdd_ReadVariableOp_shape-[2]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_005_sequential_dense_MatMul_shape-[ 256 1600]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_006_sequential_dense_1_MatMul_shape-[256 256]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_007_sequential_dense_2_MatMul_shape-[256 256]_dtype-<class 'numpy.float32'>.npy
Saved: tensor_008_sequential_y_pred_MatMul_shape-[  2 256]_dtype-<class 'numpy.float32'>.npy
Failed to extract sequential/dense/MatMul;sequential/dense/Relu;sequential/dense/BiasAdd (index 9): Tensor data is n

[3] Code to inspect the files saved to see if the values are plausible. Only a check block

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

def verify_npy_files(directory="tflite_tensors"):
    """Verify contents of .npy files in a directory."""
    npy_files = [f for f in os.listdir(directory) if f.endswith('.npy')]

    if not npy_files:
        print(f"No .npy files found in {directory}!")
        return

    print(f"Found {len(npy_files)} .npy files in {directory}:\n")

    for file in sorted(npy_files):
        filepath = os.path.join(directory, file)
        data = np.load(filepath)

        print(f"{file}:")
        print(f"  - Shape: {data.shape}")
        print(f"  - Dtype: {data.dtype}")
        print(f"  - Min: {np.min(data):.4f}, Max: {np.max(data):.4f}, Mean: {np.mean(data):.4f}")
        print(f"  - Size: {data.size} elements\n")

        if len(data.shape) == 4 and 'conv' in file.lower():
            print("  Visualizing first filter of 4D weights...")
            plt.figure(figsize=(3, 3))
            plt.imshow(data[0, :, :, 0], cmap='viridis')
            plt.colorbar()
            plt.title(f"{file} (first filter)")
            plt.show()

if __name__ == "__main__":
    verify_npy_files()


Found 10 .npy files in tflite_tensors:

tensor_000_serving_default_x:0_shape-[   1 1600]_dtype-<class 'numpy.float32'>.npy:
  - Shape: (1, 1600)
  - Dtype: float32
  - Min: nan, Max: nan, Mean: nan
  - Size: 1600 elements

tensor_001_sequential_dense_1_BiasAdd_ReadVariableOp_shape-[256]_dtype-<class 'numpy.float32'>.npy:
  - Shape: (256,)
  - Dtype: float32
  - Min: -0.0550, Max: 0.0899, Mean: 0.0118
  - Size: 256 elements

tensor_002_sequential_dense_2_BiasAdd_ReadVariableOp_shape-[256]_dtype-<class 'numpy.float32'>.npy:
  - Shape: (256,)
  - Dtype: float32
  - Min: -0.0446, Max: 0.1062, Mean: 0.0170
  - Size: 256 elements

tensor_003_sequential_dense_BiasAdd_ReadVariableOp_shape-[256]_dtype-<class 'numpy.float32'>.npy:
  - Shape: (256,)
  - Dtype: float32
  - Min: -0.0492, Max: 0.1005, Mean: 0.0175
  - Size: 256 elements

tensor_004_sequential_y_pred_BiasAdd_ReadVariableOp_shape-[2]_dtype-<class 'numpy.float32'>.npy:
  - Shape: (2,)
  - Dtype: float32
  - Min: -0.0450, Max: 0.0450, M

[4] Header Creation to save the arrays

In [ ]:
import numpy as np

# Function to sanitize variable names
def sanitize_name(name):
    return name.replace('/', '_').replace(';', '_').replace(':', '_')

# Function to extract the last part of the name
def clear_name(name):
    parts = name.rsplit(";")
    return parts[-1]

npy_dir="tflite_tensors"
npy_files = [f for f in os.listdir(npy_dir) if f.endswith('.npy')]

# 1. Generate the C header file
with open("model_weights.h", "w") as f:
    f.write("#ifndef MODEL_WEIGHTS_H\n")
    f.write("#define MODEL_WEIGHTS_H\n\n")
    #2. Accessing to each file that corresponds to an array
    for file in sorted(npy_files):
        data = np.load(os.path.join(npy_dir, file))
        if data.size == 0:
            continue

        tensor_name = "_".join(file.split('_')[2:-2])
        c_name = sanitize_name(tensor_name)
        # 3. Data printing format and definition
        f.write(f"static const float {c_name}[{data.size}] = {{\n")

        flat_data = data.flatten()
        for i in range(0, len(flat_data), 8):
            line = ", ".join(f"{x:.6f}f" for x in flat_data[i:i+8])
            f.write(f"    {line},\n")

        f.write("};\n\n")
        f.write(f"// Shape: {data.shape}\n\n")

    f.write("#endif // MODEL_WEIGHTS_H\n")
